# 04 — USB Autorun Payload Simulation

| Field | Value |
|-------|-------|
| **MITRE ATT&CK ID** | T1091 — Replication Through Removable Media |
| **Tactic** | Initial Access |
| **Difficulty** | Beginner |
| **Estimated Time** | 30 minutes |

## Threat Context: Stuxnet — USB as a Weapon

Stuxnet, discovered in 2010, was a nation-state cyberweapon designed to sabotage Iran's nuclear centrifuges. It propagated via infected USB drives, exploiting Windows autorun functionality and zero-day vulnerabilities. An operator would unknowingly carry an infected USB into the air-gapped Natanz facility, and the malware would execute automatically upon insertion.

USB-based attacks remain relevant: the **Raspberry Robin** worm (2022) spreads via USB drives containing malicious `.lnk` files, and **BadUSB** attacks reprogram USB firmware to emulate keyboards. These techniques bypass network-based defenses entirely.

## What You'll Understand After This

- How **autorun.inf** files and disguised executables work on removable media
- The **attack chain** from USB insertion to payload execution
- How to **detect and prevent** USB-based initial access through Group Policy, endpoint controls, and monitoring

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Environment setup and imports

import os
import shutil
import tempfile
import json
from pathlib import Path
from datetime import datetime

print("[+] Imports loaded successfully")
print("[i] This notebook simulates USB payload structure in a TEMP directory.")
print("[i] No actual USB devices or autorun exploits are used.")

### Step 1 — Understand the Autorun.inf Structure

Windows historically used `autorun.inf` files to automatically execute programs when a USB drive was inserted. Although modern Windows versions (7+) disable autorun for USB drives by default, attackers adapted by using social engineering (e.g., disguised shortcuts, fake document icons).

In [ ]:
# EDUCATIONAL PURPOSE ONLY — demonstrate autorun.inf structure

# Create a temporary directory to simulate a USB drive
usb_sim_dir = tempfile.mkdtemp(prefix="cyberlab_usb_")
print(f"[+] Simulated USB root: {usb_sim_dir}")

# Classic autorun.inf content (this format was used pre-Windows 7)
autorun_content = """[autorun]
open=setup.exe
icon=drive.ico
label=Company Documents
action=Open folder to view files
"""

autorun_path = os.path.join(usb_sim_dir, "autorun.inf")
with open(autorun_path, "w") as f:
    f.write(autorun_content)

print("[+] Created autorun.inf:")
print(autorun_content)
print("[i] Key fields:")
print("    open=    -> executable to run automatically")
print("    icon=    -> custom drive icon (social engineering)")
print("    label=   -> custom drive label")
print("    action=  -> text shown in autoplay dialog")

### Step 2 — Simulate a Social Engineering USB Layout

Modern USB attacks rely on social engineering: the drive appears to contain enticing documents, but the files are actually disguised payloads (e.g., `.lnk` shortcuts, files with double extensions like `report.pdf.exe`).

In [ ]:
# EDUCATIONAL PURPOSE ONLY — simulate a social engineering USB layout

# Create folder structure that looks legitimate
dirs = [
    os.path.join(usb_sim_dir, "Company Documents"),
    os.path.join(usb_sim_dir, "Photos"),
    os.path.join(usb_sim_dir, ".hidden"),  # Hidden payload directory
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

# Create decoy files
decoy_files = {
    "Company Documents/Q4_Report.pdf.txt": "[This simulates a disguised payload with double extension]",
    "Company Documents/Budget_2024.xlsx.txt": "[Simulated decoy spreadsheet]",
    "Photos/team_photo.jpg.txt": "[Simulated image file]",
    ".hidden/payload_sim.txt": "# EDUCATIONAL: This represents a hidden payload\nprint('Simulated payload executed')",
}

created_files = []
for rel_path, content in decoy_files.items():
    full_path = os.path.join(usb_sim_dir, rel_path)
    with open(full_path, "w") as f:
        f.write(content)
    created_files.append(rel_path)
    print(f"  [+] Created: {rel_path}")

print(f"\n[+] USB layout created with {len(created_files)} files")
print("[i] Attacker technique: double extensions trick users into opening executables")

### Step 3 — Map the USB Attack Chain

Let's document the complete attack chain from USB creation to payload execution. This helps defenders understand where detection is possible.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — document the attack chain

attack_chain = [
    {
        "phase": "1. Preparation",
        "attacker_action": "Create USB with payload + social engineering files",
        "technique": "T1091, T1204 (User Execution)",
        "detection_opportunity": "USB device inventory, endpoint DLP"
    },
    {
        "phase": "2. Delivery",
        "attacker_action": "Drop USB in parking lot, mail USB as 'gift'",
        "technique": "Physical social engineering",
        "detection_opportunity": "Security awareness training"
    },
    {
        "phase": "3. Insertion",
        "attacker_action": "Target inserts USB into workstation",
        "technique": "T1091",
        "detection_opportunity": "USB device control policy, EDR alerts"
    },
    {
        "phase": "4. Execution",
        "attacker_action": "Autorun or user opens disguised payload",
        "technique": "T1204.002 (Malicious File)",
        "detection_opportunity": "Application whitelisting, behavioral analysis"
    },
    {
        "phase": "5. Persistence",
        "attacker_action": "Payload establishes persistence on host",
        "technique": "T1547 (Boot/Logon Autostart)",
        "detection_opportunity": "Registry monitoring, startup folder auditing"
    },
]

print("=" * 70)
print("USB ATTACK CHAIN")
print("=" * 70)
for step in attack_chain:
    print(f"\n{step['phase']}")
    print(f"  Action:    {step['attacker_action']}")
    print(f"  Technique: {step['technique']}")
    print(f"  Detect:    {step['detection_opportunity']}")
print("\n" + "=" * 70)

### Visualization — USB Attack Chain Flowchart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis("off")
ax.set_title("USB Attack Chain — From Creation to Execution", fontsize=14, fontweight="bold")

# Attack chain boxes
boxes = [
    (0.5, 3, "Prepare\nUSB Payload", "#e74c3c"),
    (2.5, 3, "Deliver\n(Drop/Mail)", "#e67e22"),
    (4.5, 3, "Target\nInserts USB", "#f39c12"),
    (6.5, 3, "Payload\nExecutes", "#c0392b"),
    (8.5, 3, "Persistence\nEstablished", "#8e44ad"),
]

for x, y, label, color in boxes:
    rect = mpatches.FancyBboxPatch((x, y - 0.5), 1.5, 1.2, boxstyle="round,pad=0.1",
                                   facecolor=color, edgecolor="white", alpha=0.85)
    ax.add_patch(rect)
    ax.text(x + 0.75, y + 0.1, label, ha="center", va="center",
            fontsize=9, color="white", fontweight="bold")

# Arrows
for i in range(len(boxes) - 1):
    ax.annotate("", xy=(boxes[i + 1][0], 3.1), xytext=(boxes[i][0] + 1.5, 3.1),
                arrowprops=dict(arrowstyle="->", color="#2c3e50", lw=2))

# Detection points
detections = [
    (1.25, 1.5, "Device\nInventory"),
    (3.25, 1.5, "Security\nAwareness"),
    (5.25, 1.5, "USB Device\nControl"),
    (7.25, 1.5, "App\nWhitelisting"),
    (9.25, 1.5, "EDR\nMonitoring"),
]

for x, y, label in detections:
    rect = mpatches.FancyBboxPatch((x - 0.55, y - 0.4), 1.1, 0.8, boxstyle="round,pad=0.05",
                                   facecolor="#27ae60", edgecolor="white", alpha=0.75)
    ax.add_patch(rect)
    ax.text(x, y, label, ha="center", va="center", fontsize=7, color="white")

ax.text(0.5, 5.5, "Red = Attack Steps", fontsize=10, color="#e74c3c")
ax.text(0.5, 5.0, "Green = Detection Opportunities", fontsize=10, color="#27ae60")

plt.tight_layout()
plt.show()
print("[+] Visualization complete")

## Defender's Perspective — Indicators of Compromise

| Indicator | Type | Detection Method |
|-----------|------|------------------|
| New USB device connection event | Device | Windows Event ID 2003/2100, Sysmon Event 6 |
| `autorun.inf` file on removable media | File | Endpoint DLP scanning, real-time file monitoring |
| Executable files on USB with document icons | File | Static analysis, file-type validation |
| Hidden directories on removable media | File | Disk forensics, dir /ah scanning |
| Process launched from removable drive path | Behavioral | EDR process tree monitoring, application whitelisting |

## Article Seed

**Title:** *Plugged In, Hacked Out: Why USB Attacks Remain a Persistent Threat*

**Opening Paragraph:**
A USB drive left in a parking lot. A promotional flash drive mailed to an executive. A charging cable with a hidden implant. Despite decades of security awareness campaigns, USB-based attacks continue to breach organizations by exploiting the physical trust boundary that no firewall can protect.

**Section Headers:**
1. From Stuxnet to Raspberry Robin: The Evolution of USB Attacks
2. The Social Engineering Layer: Why People Still Plug In Unknown Devices
3. Technical Deep Dive: Autorun, LNK Files, and BadUSB
4. Implementing a USB Security Policy That Actually Works

**Medium Tags:** `cybersecurity`, `usb-security`, `social-engineering`, `mitre-attack`, `initial-access`

In [ ]:
# Self-check assertions

# 1. Verify simulated USB structure was created
assert os.path.exists(autorun_path), "autorun.inf should exist in simulated USB"

# 2. Verify attack chain was documented
assert len(attack_chain) == 5, "Attack chain should have 5 phases"

# 3. Verify all decoy files were created
for rel_path in created_files:
    full = os.path.join(usb_sim_dir, rel_path)
    assert os.path.exists(full), f"Decoy file should exist: {rel_path}"

# Cleanup
shutil.rmtree(usb_sim_dir, ignore_errors=True)
print("[+] All self-check assertions passed!")
print("[+] Temporary files cleaned up.")